# L07 · Puerta trasera y confusor latente

**S7 · Lectura: Cap. 8 · 40–55 min**

## La pregunta

¿Qué hacemos cuando falta una causa común?

## Qué harás

Aplicar la puerta trasera y saber cuándo abstenerse.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Una decisión de identificación y una sensibilidad al confusor oculto.

> **Idea clave:** Un modelo no puede recuperar desde observables una variable que no fue medida.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
rng = setup()

def dag(nodos, aristas):
    """Matriz de adyacencia con pesos: aristas = {(a, b): peso} significa a → b. `nodos` en orden causal."""
    A = np.zeros((len(nodos), len(nodos)))
    for (a, b), w in aristas.items():
        A[nodos.index(a), nodos.index(b)] = w
    return A

def simular_sem(nodos, aristas, rng, n=20000):
    datos = {}
    for v in nodos:
        datos[v] = sum(w * datos[a] for (a, b), w in aristas.items() if b == v) + rng.normal(size=n)
    return pd.DataFrame(datos)

def efecto(df, controles, t="T", y="Y"):
    """Coeficiente de T en la regresión de Y sobre T y los controles."""
    return ols(df[y], df[[t, *controles]])["coef"][1]

## 1. El criterio de puerta trasera (§8.1)
Un conjunto $Z$ identifica el efecto de $T$ sobre $Y$ si
1. **ningún** elemento de $Z$ es descendiente de $T$, y
2. $Z$ **bloquea todos los caminos de puerta trasera** (los que empiezan con una flecha *hacia* $T$).

Operativamente: quita las flechas que **salen** de $T$ y comprueba que $T\perp Y\mid Z$ ($d$-separación). La función `dsep` ya está en `lab_common`.

In [ ]:
# Mundo de juguete: C = confusor, M = mediador, K = colisionador
nodos = ["C", "T", "M", "Y", "K"]
aristas = {("C", "T"): 1.0, ("C", "Y"): 1.5, ("T", "M"): 1.0, ("M", "Y"): 0.5, ("T", "Y"): 0.5, ("T", "K"): 1.0, ("Y", "K"): 1.0}
A = dag(nodos, aristas)       # efecto total de T sobre Y = 0,5 + 1,0·0,5 = 1,0
print("efecto total verdadero de T sobre Y =", 0.5 + 1.0 * 0.5)

In [ ]:
# ✏️ Completa tú — satisface_puerta_trasera
# Pista: dsep(A, xs, ys, zs) trabaja con índices; copia A y pon a cero la fila de T.
def descendientes(A, i):
    """Conjunto de índices de los descendientes de i (sin incluirlo)."""
    out, pila = set(), [i]
    while pila:
        v = pila.pop()
        for w in range(len(A)):
            if A[v][w] != 0 and w not in out:
                out.add(w); pila.append(w)
    return out

def satisface_puerta_trasera(A, nodos, t, y, Z):
    """True si Z cumple el criterio de puerta trasera para T → Y en el DAG A."""
    raise NotImplementedError("(1) Z sin descendientes de T; (2) en A sin las flechas que salen de T, dsep(A', [T], [Y], Z)")

In [ ]:
conjuntos = [[], ["C"], ["M"], ["C", "M"], ["K"], ["C", "K"]]
res = [{"controles": "{" + ", ".join(Z) + "}", "puerta trasera": satisface_puerta_trasera(A, nodos, "T", "Y", Z)} for Z in conjuntos]
show(pd.DataFrame(res))
lab_check("puerta trasera", lambda: [satisface_puerta_trasera(A, nodos, "T", "Y", Z) for Z in conjuntos],
          [((), [False, True, False, False, False, False], 0)])

## 2. Buenos y malos controles: se *miden*, no se discuten
Simulamos el SEM y estimamos el efecto de $T$ con cada conjunto de controles. Compara con el efecto total verdadero (1,0).

In [ ]:
df = simular_sem(nodos, aristas, np.random.default_rng(0))
filas = []
for Z in conjuntos:
    filas.append({"controles": "{" + ", ".join(Z) + "}", "efecto estimado": efecto(df, Z),
                  "criterio dice": "válido" if satisface_puerta_trasera(A, nodos, "T", "Y", Z) else "NO válido"})
tab = pd.DataFrame(filas); tab["sesgo"] = tab["efecto estimado"] - 1.0
show(tab.round(3))

**Mira el resultado.** ¿Qué hace cada control malo? ¿Por qué ajustar por *más* variables no es más seguro?

**Respuesta:**

`{}` deja abierta la puerta trasera por **C** (sesgo hacia arriba). `{M}` o `{C, M}` ajustan por un **mediador**: bloquean parte del efecto (queda el directo, 0,5). `{K}` ajusta por un **colisionador**: abre un camino espurio entre T e Y. Solo `{C}` da ≈ 1,0.



### M-bias: un control «pre-tratamiento» que perjudica
`Z` se mide antes del tratamiento y está correlacionada con $T$ y con $Y$… pero es un **colisionador** de dos causas no medidas. Ajustar por `Z` *crea* el sesgo.

In [ ]:
nodos2 = ["U1", "U2", "Z", "T", "Y"]
aristas2 = {("U1", "Z"): 1.0, ("U2", "Z"): 1.0, ("U1", "T"): 1.0, ("U2", "Y"): 1.0, ("T", "Y"): 1.0}
A2 = dag(nodos2, aristas2); d2 = simular_sem(nodos2, aristas2, np.random.default_rng(1))
print(f"sin ajustar : {efecto(d2, []):+.3f}   (verdad = +1.000)")
print(f"ajustando Z : {efecto(d2, ['Z']):+.3f}")
print("criterio de puerta trasera: {} →", satisface_puerta_trasera(A2, nodos2, "T", "Y", []), "| {Z} →", satisface_puerta_trasera(A2, nodos2, "T", "Y", ["Z"]))

## 3. Puerta delantera (§8.3)
Si el confusor $U$ **no se mide**, no hay conjunto de puerta trasera. Pero si todo el efecto pasa por un mediador $M$ que $U$ no toca, se puede encadenar dos efectos identificables: $T\to M$ (sin confusión) y $M\to Y$ (ajustando por $T$):
$$\text{efecto}=\underbrace{a}_{T\to M}\times\underbrace{b}_{M\to Y\mid T}$$

In [ ]:
nodos3 = ["U", "T", "M", "Y"]
aristas3 = {("U", "T"): 1.0, ("U", "Y"): 1.0, ("T", "M"): 0.8, ("M", "Y"): 0.7}
d3 = simular_sem(nodos3, aristas3, np.random.default_rng(2))
a = ols(d3["M"], d3[["T"]])["coef"][1]
b = ols(d3["Y"], d3[["M", "T"]])["coef"][1]
print(f"ingenuo (U no medido)  : {efecto(d3, []):+.3f}")
print(f"puerta delantera a·b   : {a * b:+.3f}   (verdad = 0.8 · 0.7 = {0.8 * 0.7:+.3f})")

## 4. Contrafactual: $Y=2X+U$ (ejercicio del capítulo)
Una unidad tiene $X=1$ y $Y=3$. ¿Qué habría pasado con $X=2$? Abducción: $U=Y-2X=1$. Predicción: $Y'=2\cdot2+1=5$.

In [ ]:
x, y_obs = 1.0, 3.0
u = y_obs - 2 * x
print("U =", u, "· Y con X=2 →", 2 * 2.0 + u)

## 5. Cuando **no** hay conjunto válido: un confusor latente
El escenario `latent_confounder`: el tratamiento (X8) y el resultado (X9) comparten una causa **no medida** (X0). Analista: ve X1–X7, X8 y X9.

Primero lo que dice el grafo. Sin importar cómo se conecten X1–X7, el camino `X8 ← X0 → X9` sigue abierto porque X0 no se puede condicionar. Lo comprobamos enumerando **todos** los subconjuntos observados.

In [ ]:
from itertools import combinations
sc = load_scenario("synthetic/edge_cases/latent_confounder"); sc.card()
nd = ["X0"] + [f"X{i}" for i in range(1, 10)]                 # X0 es latente
A_l = dag(nd, {("X0", "X8"): 1, ("X0", "X9"): 1, ("X8", "X9"): 1})
obs = [f"X{i}" for i in range(1, 8)]
validos = [Z for k in range(len(obs) + 1) for Z in combinations(obs, k) if satisface_puerta_trasera(A_l, nd, "X8", "X9", list(Z))]
print("subconjuntos observados que cumplen el criterio:", len(validos), "de", 2 ** len(obs))

In [ ]:
# Vista rápida de los datos del escenario
import matplotlib.pyplot as plt
import numpy as np
_fig, _ax = plt.subplots(1, 2, figsize=(9, 3.2))
try:
    _scenario = globals().get("sc") or globals().get("lc")
    if _scenario is None:
        raise NameError("No hay un escenario cargado para el gráfico editorial")
    _df = _scenario.train
    if _scenario.treatment is not None and _scenario.outcome is not None:
        _t = _df[_scenario.treatment].to_numpy(float)
        _y = _df[_scenario.outcome].to_numpy(float)
        if np.unique(_t).size == 2:
            _counts = [int((_t == value).sum()) for value in (0, 1)]
            _ax[0].bar(["control", "tratado"], _counts)
            _ax[0].set_title("Asignación")
            _ax[1].boxplot([_y[_t == 0], _y[_t == 1]], labels=["control", "tratado"])
            _ax[1].set_title("Resultado observado")
        else:
            _ax[0].hist(_t, bins=20)
            _ax[0].set_title(_scenario.treatment)
            _ax[1].scatter(_t, _y, s=8, alpha=.35)
            _ax[1].set_title(f"{_scenario.treatment} frente a {_scenario.outcome}")
    elif _scenario.adj is not None:
        _ax[0].imshow(_scenario.adj, cmap="Greys", interpolation="nearest")
        _ax[0].set_title("Grafo del escenario")
        _ax[1].axis("off")
    else:
        _ax[0].axis("off"); _ax[1].axis("off")
    _fig.suptitle(f"Escenario: {_scenario.name}")
    _fig.tight_layout(); plt.show()
finally:
    plt.close(_fig)

In [ ]:
X, y, t = sc.xyt("train")
cov = sc.covariates
b_naive = ols(y, t)["coef"][1]; r_adj = ols(y, np.column_stack([t, X])); b_adj = r_adj["coef"][1]
print(f"pendiente de X8 sobre X9 sin ajustar          = {b_naive:+.3f}")
print(f"ajustando por todo lo observado (X1–X7)       = {b_adj:+.3f}  (IC95% {ci95(b_adj, r_adj['se'][1])[0]:+.3f} a {ci95(b_adj, r_adj['se'][1])[1]:+.3f})")

## Sección de evaluación reservada

La edición pública termina antes de las celdas docentes que comparan con verdad simulada, efectos potenciales o resultados oráculo. Esa revisión se ejecuta aparte; aquí debes sostener la decisión con observables, incertidumbre y supuestos declarados.
